In [ ]:
# ------ CELL 1: INSTALLATION ------------------------------------------------------------------------------
# Estimated time: 2-3 min. Wait for DONE.

import os, shutil, subprocess, sys

# 1. Install system dependencies
!apt-get install -y zstd curl lsof -q 2>/dev/null

# 2. Install Ollama
OLLAMA_PATHS = [
    shutil.which("ollama"),
    "/usr/local/bin/ollama",
    "/usr/bin/ollama",
    os.path.expanduser("~/.local/bin/ollama"),
    os.path.expanduser("~/ollama"),
]
if not any(p and os.path.exists(p) for p in OLLAMA_PATHS):
    !curl -fsSL https://ollama.com/install.sh | sh
else:
    print("  Ollama already installed")

# 3. Install Open WebUI
try:
    import open_webui
    print("  Open WebUI already installed")
except ImportError:
    !pip install open-webui -q

# 4. Verify
missing = []
if not shutil.which("ollama"):
    ollama_found = any(p and os.path.exists(p) for p in OLLAMA_PATHS)
    if not ollama_found:
        missing.append("ollama")
try:
    import open_webui
except ImportError:
    missing.append("open-webui")

if missing:
    print(f"FAILED: {', '.join(missing)} not found")
else:
    print("=============================================")
    print("DONE - Proceed to Cell 2.")
    print("=============================================")


In [ ]:
# ------ CELL 2: SETUP & RUN OLLAMA ---------------------------------------------------------------
# Change model below if needed:
MODEL = "qwen2.5-coder:7b"   # default (~4.7GB)
# MODEL = "qwen2.5-coder:14b"  # coding (~9.0GB)
# MODEL = "huihui_ai/qwen2.5-coder-abliterate:14b-instruct-q4_k_m"  # uncensored coding (~9.0GB)
# MODEL = "supergoatscriptguy/mythos-sec:24b"  # pentest/CTF uncensored (~14GB)
# MODEL = "dolphin-llama3:8b"  # uncensored general/agentic (~4.7GB)
# MODEL = "deepseek-coder:14b" # coding (~8.5GB)
# MODEL = "deepseek-r1:7b"     # reasoning (~4.7GB)
# MODEL = "llama3.1:8b"        # general chat (~4.9GB)
# MODEL = "gemma4:12b"         # reasoning & coding (~6.6GB)

import subprocess, time, os, shutil, json, urllib.request, sys
from google.colab import drive

# --- Find ollama binary ---
OLLAMA_CANDIDATES = [
    shutil.which("ollama"),
    "/usr/local/bin/ollama",
    "/usr/bin/ollama",
    os.path.expanduser("~/.local/bin/ollama"),
    os.path.expanduser("~/ollama"),
]
ollama_bin = next((p for p in OLLAMA_CANDIDATES if p and os.path.exists(p)), None)
if not ollama_bin:
    print("Ollama not found. Run Cell 1 first.")
    sys.exit(1)

# --- Kill stale Ollama processes ---
os.system("pkill -f 'ollama serve' 2>/dev/null || true")
time.sleep(1)

# --- Start Ollama ---
print("[1/4] Starting Ollama...")
os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
proc_ollama = subprocess.Popen(
    [ollama_bin, "serve"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)

print("  Waiting for Ollama...", end="", flush=True)
for _ in range(30):
    try:
        r = urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
        r.close()
        print(" OK")
        ollama_ready = True
        break
    except Exception:
        print(".", end="", flush=True)
        time.sleep(2)
if not ollama_ready:
    print(" FAILED")
    sys.exit(1)

# --- Pull model ---
print("[2/4] Pulling model...")
pull = subprocess.Popen(
    [ollama_bin, "pull", MODEL],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)
for line in iter(pull.stdout.readline, ''):
    line = line.rstrip()
    if line:
        print(f"  {line}")
pull.wait()
if pull.returncode != 0:
    print(f"  Pull failed (exit {pull.returncode})")
    sys.exit(1)

# --- Mount Drive & backup ---
print("[3/4] Mounting Drive...")
if not os.path.exists("/content/drive"):
    drive.mount('/content/drive')

print("[4/4] Backing up to Drive...")
LOCAL_PATH = os.path.expanduser("~/.ollama/models")
DRIVE_PATH = "/content/drive/MyDrive/ollama_models"
if os.path.exists(LOCAL_PATH):
    os.makedirs(DRIVE_PATH, exist_ok=True)
    for item in os.listdir(LOCAL_PATH):
        if item.startswith('.'):
            continue
        src = os.path.join(LOCAL_PATH, item)
        dst = os.path.join(DRIVE_PATH, item)
        if not os.path.exists(dst):
            try:
                if os.path.isdir(src):
                    shutil.copytree(src, dst, dirs_exist_ok=True)
                else:
                    shutil.copy2(src, dst)
            except Exception as e:
                print(f"    Backup failed: {e}")

print("=============================================")
print(f"MODEL {MODEL} READY - Proceed to Cell 3.")
print("=============================================")


In [ ]:
# ------ CELL 3: WEBUI + DIRECT OLLAMA TUNNEL ---------------------------------------
# TWO tunnels: WebUI (port 8081) + Ollama direct (port 11434)
# - WebUI tunnel -> browser access + Anthropic/Claude Code API
# - Ollama tunnel -> direct OpenAI-compatible API for opencode etc.
# MUST KEEP RUNNING while using any tunnel.

import os, subprocess, time, urllib.request, re, shutil, sys, json

WEBUI_PORT = 8081
OLLAMA_PORT = 11434

# --- Helper: kill process on a port ---
def kill_port(port):
    for cmd in [f"fuser -k {port}/tcp 2>/dev/null", f"lsof -ti:{port} | xargs kill -9 2>/dev/null"]:
        ret = os.system(cmd)
        if os.WIFEXITED(ret) and os.WEXITSTATUS(ret) == 0:
            return True
    return False

# --- Helper: start a Cloudflare tunnel and return (process, url) ---
def start_tunnel(target_port, label):
    proc = subprocess.Popen(
        [CLOUDFLARED_BIN, "tunnel", "--url", f"http://127.0.0.1:{target_port}", "--no-autoupdate"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    url = None
    print(f"  Starting {label} tunnel...", end="", flush=True)
    for _ in range(30):
        line = proc.stdout.readline()
        if not line: break
        m = re.search(r'(https://[a-zA-Z0-9.-]+\.trycloudflare\.com)', line)
        if m:
            url = m.group(1)
            print(" OK")
            break
        time.sleep(1)
        print(".", end="", flush=True)
    if not url:
        print(" FAILED")
    return proc, url

# ============================================================
# 1. START OPEN WEBUI
# ============================================================
webui_bin = shutil.which("open-webui") or shutil.which("open_webui")
if not webui_bin:
    print("open-webui not found. Run Cell 1 first.")
    raise SystemExit(1)

print("[1/4] Cleaning ports...", flush=True)
kill_port(WEBUI_PORT)
time.sleep(1)

print("[2/4] Starting Open WebUI...", flush=True)
env = {**os.environ, "OLLAMA_BASE_URL": "http://127.0.0.1:11434", "WEBUI_AUTH": "False"}
proc_webui = subprocess.Popen(
    [webui_bin, "serve", "--port", str(WEBUI_PORT)],
    env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)

print("  Waiting for WebUI (1-3 min)...", flush=True)
for i in range(60):
    if proc_webui.poll() is not None:
        print(f"  WebUI failed (exit {proc_webui.returncode})")
        raise SystemExit(1)
    try:
        r = urllib.request.urlopen(f"http://127.0.0.1:{WEBUI_PORT}", timeout=5)
        r.close()
        print("  WebUI OK")
        break
    except Exception:
        print(".", end="", flush=True)
        time.sleep(5)
else:
    print("  WebUI timeout")

# ============================================================
# 2. CLOUDFLARE TUNNELS
# ============================================================

# --- Download cloudflared once ---
CLOUDFLARED_BIN = "/usr/local/bin/cloudflared"
if not os.path.exists(CLOUDFLARED_BIN):
    print("[3/4] Downloading cloudflared...", flush=True)
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        CLOUDFLARED_BIN
    )
    os.chmod(CLOUDFLARED_BIN, 0o755)
else:
    print("[3/4] cloudflared ready")

os.system("pkill -f cloudflared 2>/dev/null || true")
time.sleep(1)

# --- Tunnel A: WebUI (port 8081) for browser + Anthropic API ---
proc_cf_webui, webui_url = start_tunnel(WEBUI_PORT, "WebUI")
if not webui_url:
    sys.exit(1)

# --- Tunnel B: Ollama direct (port 11434) for OpenAI-compatible API ---
proc_cf_ollama, ollama_tunnel_url = start_tunnel(OLLAMA_PORT, "Ollama")
if not ollama_tunnel_url:
    sys.exit(1)

# ============================================================
# 3. TESTS
# ============================================================
print("[4/4] Testing...")
ollama_api_v1 = f"{ollama_tunnel_url}/v1"
ollama_native = f"{webui_url}/ollama"

try:
    # Test A1: Ollama native API via WebUI proxy
    r = urllib.request.urlopen(f"{ollama_native}/api/tags", timeout=15)
    tags = json.loads(r.read())
    r.close()
    models_found = [m['name'] for m in tags.get('models', [])]
    print(f"  Ollama native: {len(models_found)} models")
    if MODEL in models_found:
        print(f"    '{MODEL}' is ready")
    else:
        print(f"    WARNING: '{MODEL}' not found in Ollama")

    # Test A2: OpenAI-compatible via DIRECT Ollama tunnel
    r = urllib.request.urlopen(f"{ollama_api_v1}/models", timeout=15)
    r.close()
    print(f"  Ollama direct /v1/models: OK")

    # Test A3: POST /v1/chat/completions via DIRECT Ollama tunnel
    body = json.dumps({
        "model": MODEL,
        "messages": [{"role": "user", "content": "Say OK"}],
        "stream": False,
        "max_tokens": 10
    }).encode()
    req = urllib.request.Request(
        f"{ollama_api_v1}/chat/completions",
        data=body,
        headers={"Content-Type": "application/json"},
        method="POST"
    )
    r = urllib.request.urlopen(req, timeout=120)
    resp = json.loads(r.read())
    r.close()
    reply = resp['choices'][0]['message']['content']
    print(f"  Ollama direct chat test: \"{reply}\"")
    print("  API fully functional")

except json.JSONDecodeError as e:
    print(f"  FAILED - Invalid JSON response: {e}")
    sys.exit(1)
except Exception as e:
    print(f"  FAILED - {e}")
    sys.exit(1)

print()

# ============================================================
# 4. PRINT CONFIGURATION
# ============================================================
SEP = "=" * 60
print(SEP)
print("---- WEBUI (open in browser):")
print(f"  {webui_url}")
print(SEP)
print()

# ------ opencode config (uses DIRECT Ollama tunnel) ---------------------------
print(SEP)
print("---- OPENCODE CONFIG (uses Ollama direct tunnel):")
print(f"  endpoint: {ollama_api_v1}")
print(SEP)
print("Add this to opencode.json:")
print("{")
print('  "$schema": "https://opencode.ai/config.json",')
print('  "provider": {')
print('    "Ollama": {')
print('      "npm": "@ai-sdk/openai-compatible",')
print('      "name": "Ollama (Colab)",')
print('      "options": {')
print(f'        "baseURL": "{ollama_api_v1}"'),
print('      },')
print('      "models": {')
print(f'        "{MODEL}": {{')
print(f'          "name": "{MODEL}"')
print('        }')
print('      }')
print('    }')
print('  }')
print("}")
print()

# ------ Claude Code Router config (uses WebUI Anthropic API) ---
print(SEP)
print("---- CLAUDE CODE ROUTER (uses WebUI Anthropic API):")
print(SEP)
print("Set these env vars (WEBUI_AUTH=False so any key works):")
print(f'  export ANTHROPIC_BASE_URL="{webui_url}/api"')
print(f'  export ANTHROPIC_API_KEY="not-needed"')
print()
print("Or create ~/.claude/settings.json:")
print("{")
print('  "env": {')
print(f'    "ANTHROPIC_BASE_URL": "{webui_url}/api",')
print('    "ANTHROPIC_AUTH_TOKEN": "not-needed"')
print('  }')
print("}")
print()

# ------ OpenAI-compatible for generic tools (Cline, Continue, etc.) ---
print(SEP)
print("---- GENERIC OPENAI-COMPATIBLE (for Cline, Continue, etc.):")
print(SEP)
print(f"  baseURL: {ollama_api_v1}")
print(f"  model:  {MODEL}")
print()

print(SEP)
print("NOTES:")
print("  - Keep this cell running while using tunnels")
print("  - If Colab disconnects, re-run Cell 1 -> 2 -> 3")
print(f"  - Model name: {MODEL}")
print("  - Ollama direct tunnel = standard OpenAI-compatible (works with any client)")
print("  - WebUI tunnel = browser UI + Anthropic Messages API (for Claude Code)")
print()

# ============================================================
# 5. MONITOR ALL PROCESSES
# ============================================================
print("Monitoring (Ctrl+C to stop)...")
try:
    while True:
        time.sleep(30)
        now = time.strftime("%H:%M:%S")
        status = f"[{now}]"
        if proc_webui.poll() is not None:
            print("  WebUI: STOPPED")
        else:
            status += " WebUI OK"
        if proc_cf_webui.poll() is not None:
            print("  WebUI tunnel: STOPPED")
        else:
            status += " | WebUI-tunnel OK"
        if proc_cf_ollama.poll() is not None:
            print("  Ollama tunnel: STOPPED")
            break
        else:
            status += " | Ollama-tunnel OK"
        print(status)
except KeyboardInterrupt:
    print("Stopped.")
